# 13 · PPO with a multi-categorical action space (the draft plan's algorithm, as a comparison)

The draft plan proposed **PPO** with a discrete multi-categorical action. PPO is beyond the
24AIM304 syllabus, so it is kept as a *deep-RL comparison* against the tabular agents:

* observation: the **continuous** state (whitened PCA of the descriptors + location one-hot) —
  no discretisation, so the policy can in principle adapt per recording
* action: `MultiDiscrete([5, 3, 5, 4, 7])` = (wavelet, level, band, window, fusion) chosen jointly
* one-step episodes; reward = the same per-recording Δ-reward; same budget of reward queries

In [ ]:
%run common/00_config.ipynb
%run common/01_signal_lib.ipynb
%run common/02_rl_lib.ipynb
import gymnasium as gym
import matplotlib.pyplot as plt
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback

K = load_json(P["results"] / "chosen_K.json")["K"]
prob = RLProblem(CFG, P, K=K)
qc = pd.read_csv(P["metadata"] / "records_qc.csv").set_index("record_id")
train_df = qc.loc[prob.meta.record_id[prob.idx["train"]]].reset_index()
cm = ContextModel(1, CFG["context"]["n_pca"], seed=CFG["seed"]).fit(train_df)
obs = {s: cm.embed(qc.loc[prob.meta.record_id[prob.idx[s]]].reset_index()).astype(np.float32) for s in prob.idx}
dims = prob.space.dims


class ConfigSelectEnv(gym.Env):
    def __init__(self, O, R, seed=0):
        self.O, self.R = O, R
        self.observation_space = gym.spaces.Box(-np.inf, np.inf, (O.shape[1],), np.float32)
        self.action_space = gym.spaces.MultiDiscrete(dims)
        self.rng = np.random.default_rng(seed)

    def reset(self, seed=None, options=None):
        self.i = int(self.rng.integers(len(self.O)))
        return self.O[self.i], {}

    def step(self, action):
        c = prob.space.encode([int(a) for a in action])
        return self.O[self.i], float(self.R[self.i, c]), True, False, {}


class ValCurve(BaseCallback):
    def __init__(self, every=4000):
        super().__init__()
        self.every, self.rows = every, []

    def _on_step(self):
        if self.num_timesteps % self.every == 0:
            self.rows.append({"episode": self.num_timesteps, **evaluate_ppo(self.model, "val")})
        return True


def evaluate_ppo(model, split):
    a, _ = model.predict(obs[split], deterministic=True)
    c = np.array([prob.space.encode(list(map(int, r))) for r in a])
    p = prob.p[split]["clean"][c, np.arange(len(c))]
    m = binary_metrics(prob.y[split], p)
    return {"val_reward" if split == "val" else "reward": float(prob.R[split][np.arange(len(c)), c].mean()),
            "macro_f1": m["macro_f1"], "balanced_acc": m["balanced_acc"], "auroc": m["auroc"], "n_distinct_configs": len(set(c))}


curves, rows = {}, []
for seed in CFG["rl"]["seeds"]:
    env = ConfigSelectEnv(obs["train"], prob.R["train"], seed)
    model = PPO("MlpPolicy", env, n_steps=1024, batch_size=256, n_epochs=10, gamma=1.0, gae_lambda=1.0,
                learning_rate=3e-4, ent_coef=0.01, policy_kwargs={"net_arch": [64, 64]}, seed=seed, verbose=0, device="cpu")
    cb = ValCurve()
    t0 = time.time()
    model.learn(CFG["rl"]["episodes"], callback=cb)
    curves[seed] = pd.DataFrame(cb.rows)
    rows.append({"method": "PPO (continuous state, MultiDiscrete)", "seed": seed, "sec": time.time() - t0, **evaluate_ppo(model, "val")})
    model.save(P["models"] / f"ppo_config_selector_s{seed}")
ppo_tab = pd.DataFrame(rows)
ppo_tab.to_csv(P["tables"] / "13_ppo_results.csv", index=False)
ppo_tab

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for s, c in curves.items():
    ax[0].plot(c.episode, c.val_reward, label=f"seed {s}"); ax[1].plot(c.episode, c.macro_f1, label=f"seed {s}")
ax[0].axhline(0, color="gray", ls=":"); ax[0].set_title("PPO: validation mean Δ-reward"); ax[1].set_title("PPO: validation macro-F1")
for a in ax:
    a.set_xlabel("episodes = reward queries"); a.legend()
plt.tight_layout(); savefig(fig, "13_ppo_curves"); plt.show()